### Nexmark

In [ ]:
from pyflink.table import EnvironmentSettings, TableEnvironment
import os

from pathlib import Path

from pyflink.java_gateway import get_gateway
from pyflink.datastream import StreamExecutionEnvironment
from pyflink.table import StreamTableEnvironment, ExplainDetail

gateway = get_gateway()
string_class = gateway.jvm.String
string_array = gateway.new_array(string_class, 0)
stream_env = gateway.jvm.org.apache.flink.streaming.api.environment.StreamExecutionEnvironment
j_stream_exection_environment = stream_env.createRemoteEnvironment(
    "localhost", 
    8081, 
    string_array
)

env = StreamExecutionEnvironment(j_stream_exection_environment)
env.set_parallelism(2)
table_env = StreamTableEnvironment.create(env)

# Generate a proper file URI for the jar in the current directory
jar_uri = Path("flink-sql-connector-kafka-4.0.0.jar").absolute().as_uri()
print(jar_uri)

t_env=table_env
current_dir = os.getcwd()
t_env.get_config().get_configuration().set_string(
    "pipeline.jars", jar_uri
)
t_env.get_config().get_configuration().set_string("table.plan.force-recompile", "true")
# Define the source table using DDL (update the file path as needed)
source_ddl = """
CREATE TABLE bid (
        auction  BIGINT,
        bidder  BIGINT,
        price  BIGINT,
        dateTime  TIMESTAMP(3),
        extra  VARCHAR,
        WATERMARK FOR dateTime AS dateTime - INTERVAL '4' SECOND)
    WITH (
        'connector' = 'kafka',
        'topic' = 'event-demo',
        'properties.bootstrap.servers' = 'kafka-service.kafka.svc.cluster.local:9092',
        'properties.group.id' = 'nexmark',
        'scan.startup.mode' = 'earliest-offset',
        'sink.partitioner' = 'round-robin',
        'properties.metadata.max.age.ms' = '1800000',
        'format' = 'json'
      );
"""
t_env.execute_sql(source_ddl)

# Define the sink table using DDL with the print connector for debugging/output
sink_ddl = """
CREATE TABLE nexmark_q11 (
  extra VARCHAR,
  auction  BIGINT,
  bidder BIGINT,
  bid_count BIGINT,
  starttime TIMESTAMP(3),
  endtime TIMESTAMP(3)
) WITH (
        'connector' = 'kafka',
        'topic' = 'result',
        'properties.bootstrap.servers' = 'kafka-service.kafka.svc.cluster.local:9092',
        'properties.group.id' = 'nexmark',
        'value.format' = 'json'      
      );
"""
t_env.execute_sql(sink_ddl)

### do the query

file:///home/arnaud/Documents/stages/uclouvain_2025/flink-justin/notebooks/forst/nexmark/flink-sql-connector-kafka-4.0.0.jar


Py4JJavaError: An error occurred while calling o10.executeSql.
: org.apache.flink.table.api.SqlParserException: SQL parse failed. Encountered "dateTime" at line 6, column 9.
Was expecting one of:
    "CONSTRAINT" ...
    "PRIMARY" ...
    "UNIQUE" ...
    "WATERMARK" ...
    <BRACKET_QUOTED_IDENTIFIER> ...
    <QUOTED_IDENTIFIER> ...
    <BACK_QUOTED_IDENTIFIER> ...
    <BIG_QUERY_BACK_QUOTED_IDENTIFIER> ...
    <HYPHENATED_IDENTIFIER> ...
    <IDENTIFIER> ...
    <UNICODE_QUOTED_IDENTIFIER> ...
    
	at org.apache.flink.table.planner.parse.CalciteParser.parseSqlList(CalciteParser.java:81)
	at org.apache.flink.table.planner.delegation.ParserImpl.parse(ParserImpl.java:102)
	at org.apache.flink.table.api.internal.TableEnvironmentImpl.executeSql(TableEnvironmentImpl.java:784)
	at java.base/jdk.internal.reflect.DirectMethodHandleAccessor.invoke(DirectMethodHandleAccessor.java:103)
	at java.base/java.lang.reflect.Method.invoke(Method.java:580)
	at org.apache.flink.api.python.shaded.py4j.ref